In [ ]:
!pip install transformers datasets accelerate scikit-learn sentencepiece protobuf -q

In [ ]:
import pandas as pd
import numpy as np
import torch
import json
import os
import gc

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from datasets import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
df = pd.read_csv('ExioNAICS_preprocessed.csv')
df['NAICS Code'] = df['NAICS Code'].astype(str)

num_labels = df['label'].nunique()

print(f"Dataset shape: {df.shape}")
print(f"Number of classes: {num_labels}")
print(f"Label range: {df['label'].min()} to {df['label'].max()}")
df.head()

## Hyperparameter Configuration

Change `SESSION` to 1, 2, or 3 before each Colab run. Each session tests a different training strategy.

In [ ]:
SESSION = 1  # 1, 2, or 3

SESSION_CONFIGS = {
    1: {"learning_rate": 2e-5, "batch_size": 16, "weight_decay": 0.01, "name": "conservative"},
    2: {"learning_rate": 3e-5, "batch_size": 32, "weight_decay": 0.01, "name": "fast_convergence"},
    3: {"learning_rate": 1e-5, "batch_size": 16, "weight_decay": 0.1,  "name": "slow_strong_reg"},
}

config = SESSION_CONFIGS[SESSION]

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 128
NUM_EPOCHS = 15
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 3
N_FOLDS = 5
SEED = 42

print(f"=== Session {SESSION}: {config['name']} ===")
print(f"  Learning rate:  {config['learning_rate']}")
print(f"  Batch size:     {config['batch_size']}")
print(f"  Weight decay:   {config['weight_decay']}")
print(f"  Max length:     {MAX_LENGTH}")
print(f"  Epochs:         {NUM_EPOCHS} (with early stopping, patience={EARLY_STOPPING_PATIENCE})")
print(f"  Folds:          {N_FOLDS}")
print(f"  Warmup ratio:   {WARMUP_RATIO}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

folds = list(skf.split(df['clean_description'], df['label']))

for i, (train_idx, val_idx) in enumerate(folds):
    train_labels = df['label'].iloc[train_idx]
    val_labels = df['label'].iloc[val_idx]
    print(f"Fold {i+1}: train={len(train_idx)}, val={len(val_idx)}, "
          f"train classes={train_labels.nunique()}, val classes={val_labels.nunique()}")

In [ ]:
def tokenize_data(texts, labels, tokenizer, max_length):
    encodings = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors=None,
    )
    dataset = Dataset.from_dict({
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "labels": labels,
    })
    return dataset

print("Tokenization function defined.")
print(f"Using tokenizer: {MODEL_NAME}, max_length: {MAX_LENGTH}")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    top1_preds = np.argmax(logits, axis=1)

    top1_acc = accuracy_score(labels, top1_preds)
    macro_f1 = f1_score(labels, top1_preds, average='macro', zero_division=0)
    weighted_f1 = f1_score(labels, top1_preds, average='weighted', zero_division=0)

    top5_acc = np.mean([
        1 if label in np.argsort(logit)[-5:] else 0
        for logit, label in zip(logits, labels)
    ])
    top10_acc = np.mean([
        1 if label in np.argsort(logit)[-10:] else 0
        for logit, label in zip(logits, labels)
    ])

    return {
        "top1_accuracy": top1_acc,
        "top5_accuracy": top5_acc,
        "top10_accuracy": top10_acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }

print("Metrics function defined: Top-1, Top-5, Top-10 accuracy, Macro F1, Weighted F1")

In [ ]:
all_fold_results = []

for fold_idx, (train_idx, val_idx) in enumerate(folds):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold_idx+1}/{N_FOLDS} — Session {SESSION} ({config['name']})")
    print(f"{'='*60}")

    train_texts = df['clean_description'].iloc[train_idx].tolist()
    val_texts = df['clean_description'].iloc[val_idx].tolist()
    train_labels = df['label'].iloc[train_idx].tolist()
    val_labels = df['label'].iloc[val_idx].tolist()

    train_dataset = tokenize_data(train_texts, train_labels, tokenizer, MAX_LENGTH)
    val_dataset = tokenize_data(val_texts, val_labels, tokenizer, MAX_LENGTH)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_labels,
    )

    output_dir = f"./results/session_{SESSION}_fold_{fold_idx+1}"

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=config['batch_size'],
        per_device_eval_batch_size=64,
        learning_rate=config['learning_rate'],
        weight_decay=config['weight_decay'],
        warmup_ratio=WARMUP_RATIO,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="top1_accuracy",
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=50,
        fp16=True,
        seed=SEED,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    trainer.train()

    eval_results = trainer.evaluate()
    eval_results["fold"] = fold_idx + 1
    all_fold_results.append(eval_results)

    print(f"\nFold {fold_idx+1} results:")
    print(f"  Top-1 Accuracy: {eval_results['eval_top1_accuracy']:.4f}")
    print(f"  Top-5 Accuracy: {eval_results['eval_top5_accuracy']:.4f}")
    print(f"  Top-10 Accuracy: {eval_results['eval_top10_accuracy']:.4f}")
    print(f"  Macro F1:       {eval_results['eval_macro_f1']:.4f}")
    print(f"  Weighted F1:    {eval_results['eval_weighted_f1']:.4f}")

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"  ALL FOLDS COMPLETE — Session {SESSION}")
print(f"{'='*60}")

In [ ]:
metrics_keys = ["eval_top1_accuracy", "eval_top5_accuracy", "eval_top10_accuracy",
                "eval_macro_f1", "eval_weighted_f1"]

print(f"\n=== Session {SESSION} ({config['name']}) — Summary ===\n")
print(f"{'Metric':<22} {'Mean':>8} {'Std':>8}  Per-fold values")
print("-" * 75)

summary = {"session": SESSION, "config": config}
for key in metrics_keys:
    values = [r[key] for r in all_fold_results]
    mean_val = np.mean(values)
    std_val = np.std(values)
    fold_str = ", ".join([f"{v:.4f}" for v in values])
    print(f"{key:<22} {mean_val:>8.4f} {std_val:>8.4f}  [{fold_str}]")
    summary[key] = {"mean": mean_val, "std": std_val, "per_fold": values}

os.makedirs("results", exist_ok=True)
results_file = f"results/session_{SESSION}_{config['name']}.json"
with open(results_file, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\nResults saved to: {results_file}")